# Mary Johnson — European Defence Portfolio Advisory Research
## Yahoo Finance | Portfolio Construction | Diversification | CAPM | Robustness

### Role
You are the financial analyst supporting an Asset Management team advising **Mary Johnson, age 65, recently retired**.

Mary's equity wealth is concentrated in five major publicly listed **European defence companies**. The purpose of this notebook is not to argue that defence is a weak investment theme. The purpose is to test whether a concentrated defence portfolio is **efficient, sufficiently diversified, and robust enough for a retired HNWI investor**.

### Selected European defence leaders
The notebook uses five large, established, defence-focused European listed companies available through Yahoo Finance:

- **BA.L — BAE Systems plc** (United Kingdom)
- **RHM.DE — Rheinmetall AG** (Germany)
- **HO.PA — Thales S.A.** (France)
- **LDO.MI — Leonardo S.p.A.** (Italy)
- **SAAB-B.ST — Saab AB** (Sweden)

> **Scope note:** “Top five” here means a curated group of leading listed European defence contractors, not a claim that these are the five largest European aerospace/industrial companies by total market capitalization. Diversified aerospace groups such as Airbus are intentionally excluded so the portfolio remains defence-focused.

---

## Research architecture

1. Define the defence universe directly with Yahoo Finance tickers.
2. Download historical market prices, benchmark data, FX rates, and a risk-free proxy from Yahoo Finance.
3. Convert securities from EUR, GBP/GBp, and SEK into a common **USD analysis currency**.
4. Reconstruct Mary's historical **equal-number-of-shares, dividend-adjusted** defence portfolio.
5. Test whether any holding period of six years or longer lost money.
6. Compare the concentrated defence portfolio with diversified European companies.
7. Calculate annualized return, risk, Sharpe Ratio, drawdown, Sortino Ratio, and correlations.
8. Demonstrate the diversification effect with 50,000 simulated long-only portfolios.
9. Calculate CAPM Beta and Alpha against a broad European equity benchmark.
10. Add rolling-Beta robustness checks and an Investment Committee summary.

---

## Production-oriented design principles

This notebook is written as a **research prototype that could be industrialized**:

- one external market-data source: Yahoo Finance via `yfinance`;
- explicit configuration for tickers, currencies, dates, and assumptions;
- reusable functions for downloading and transforming data;
- common-currency conversion before portfolio aggregation;
- data-quality checks before calculations;
- clearly separated market-data, transformation, analytics, and decision layers.

A production implementation would replace notebook state with scheduled pipelines, persistent storage, formal data contracts, logging, tests, lineage, and monitored jobs.


## 0. Setup and configuration


In [ ]:
# If yfinance or Plotly is missing in a fresh Jupyter/Colab environment,
# uncomment the next line and run it once.
# %pip install -q yfinance plotly

from datetime import date, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# yfinance is the only external market-data interface used in this notebook.
import yfinance as yf

# Plotly is used only for an interactive comparison chart.
import plotly.express as px

# Make tables easier to read.
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
plt.rcParams["figure.figsize"] = (12, 7)

# -----------------------------
# Core model assumptions
# -----------------------------
TRADING_DAYS = 252
N_PORTFOLIOS = 50_000
SEED = 111

# The analysis starts in 2015 so that the notebook has more than
# six years of history while remaining relevant to the modern defence cycle.
START_DATE = "2015-01-01"

# yfinance treats the end date as exclusive.
# Adding one day makes the notebook request data through today's date.
END_DATE = (date.today() + timedelta(days=1)).isoformat()

# -----------------------------
# Defence portfolio universe
# -----------------------------
DEFENCE = {
    "BA.L": {
        "Company": "BAE Systems plc",
        "Country": "United Kingdom",
        "Quote_Currency": "GBp",
    },
    "RHM.DE": {
        "Company": "Rheinmetall AG",
        "Country": "Germany",
        "Quote_Currency": "EUR",
    },
    "HO.PA": {
        "Company": "Thales S.A.",
        "Country": "France",
        "Quote_Currency": "EUR",
    },
    "LDO.MI": {
        "Company": "Leonardo S.p.A.",
        "Country": "Italy",
        "Quote_Currency": "EUR",
    },
    "SAAB-B.ST": {
        "Company": "Saab AB",
        "Country": "Sweden",
        "Quote_Currency": "SEK",
    },
}

DEFENCE_TICKERS = list(DEFENCE.keys())

# -----------------------------
# Diversification candidates
# -----------------------------
# These are large European companies from different economic sectors.
# All five are euro-quoted, which makes their comparison straightforward.
DIVERSIFIERS = {
    "ASML.AS": "Technology — ASML",
    "SAN.MC": "Financials — Banco Santander",
    "SAN.PA": "Health Care — Sanofi",
    "IBE.MC": "Utilities — Iberdrola",
    "MC.PA": "Consumer Discretionary — LVMH",
}

DIVERSIFIER_TICKERS = list(DIVERSIFIERS.keys())

# Broad European equity benchmark available through Yahoo Finance.
# EXSA.DE is the iShares STOXX Europe 600 UCITS ETF (DE).
MARKET_TICKER = "EXSA.DE"

# Yahoo Finance 13-week US Treasury Bill yield proxy.
# Because this notebook evaluates returns in USD, ^IRX provides a transparent
# Yahoo-sourced risk-free approximation for the Sharpe/CAPM examples.
RF_TICKER = "^IRX"

# FX series needed to convert local-currency prices into USD.
FX_TICKERS = {
    "EUR": "EURUSD=X",
    "GBP": "GBPUSD=X",
    "SEK": "SEKUSD=X",
}

ALL_PRICE_TICKERS = DEFENCE_TICKERS + DIVERSIFIER_TICKERS + [MARKET_TICKER]
ALL_DOWNLOAD_TICKERS = ALL_PRICE_TICKERS + list(FX_TICKERS.values()) + [RF_TICKER]

print("Analysis period:", START_DATE, "to", END_DATE)
print("Defence tickers:", DEFENCE_TICKERS)


# Step 1 — Define the European defence portfolio

The original case selected companies from a local listings CSV. This version removes that dependency.

The investable defence universe is explicitly defined by **Yahoo Finance ticker**, company, country, and quote currency. That makes the input transparent and reproducible.

The five selected companies span the United Kingdom, Germany, France, Italy, and Sweden. This creates geographical diversity inside the defence theme, but it is still a **single-industry concentration**.


In [ ]:
# Convert the configuration dictionary into a DataFrame for inspection.
# Keeping configuration visible helps reviewers understand exactly
# what securities the model expects before any calculations begin.
defence_universe = (
    pd.DataFrame.from_dict(DEFENCE, orient="index")
    .rename_axis("Ticker")
    .reset_index()
)

display(defence_universe)


# Step 2 — Download all market data from Yahoo Finance

No local CSV/XLSX market-data files are required.

The notebook downloads:

- defence-company prices;
- diversification-company prices;
- a broad European equity ETF benchmark;
- EUR/USD, GBP/USD, and SEK/USD FX rates;
- the Yahoo Finance 13-week Treasury yield proxy.

We request **unadjusted Close** and **Adjusted Close**:

- `Close` is useful for reconstructing the economic value of the initial one-share positions.
- `Adj Close` incorporates Yahoo Finance's historical adjustments for dividends and corporate actions and is therefore used to estimate total-return growth.

> Yahoo Finance is suitable for research/prototyping. A production investment platform would normally use a contracted institutional data vendor with explicit SLAs, corporate-action governance, and auditability.


In [ ]:
def download_yahoo_panel(tickers, start, end):
    """
    Download Yahoo Finance daily data for all requested tickers.

    Returns
    -------
    raw : pandas.DataFrame
        MultiIndex-column DataFrame returned by yfinance.

    Production note
    ---------------
    A production service should add retries, structured logging,
    persistent raw storage, data lineage, vendor timestamp capture,
    and schema/quality monitoring around this function.
    """
    raw = yf.download(
        tickers=tickers,
        start=start,
        end=end,
        auto_adjust=False,       # keep both Close and Adj Close
        actions=False,
        progress=False,
        group_by="column",
        threads=True,
    )

    if raw.empty:
        raise RuntimeError(
            "Yahoo Finance returned no data. Check connectivity and ticker availability."
        )

    return raw


raw = download_yahoo_panel(
    tickers=ALL_DOWNLOAD_TICKERS,
    start=START_DATE,
    end=END_DATE,
)

print("Raw Yahoo shape:", raw.shape)
display(raw.tail())


In [ ]:
def extract_field(raw_panel, field):
    """
    Extract one Yahoo Finance field (for example Close or Adj Close)
    and return a simple Date x Ticker matrix.
    """
    if field not in raw_panel.columns.get_level_values(0):
        raise KeyError(f"Field '{field}' was not returned by Yahoo Finance.")

    field_df = raw_panel[field].copy()
    field_df.index = pd.to_datetime(field_df.index)
    field_df = field_df.sort_index()
    return field_df


# Separate the two price fields used by the rest of the notebook.
close_local = extract_field(raw, "Close")
adj_local = extract_field(raw, "Adj Close")

# Basic data-quality checks.
# We fail early if any required market series is completely missing.
missing_series = [
    ticker
    for ticker in ALL_DOWNLOAD_TICKERS
    if ticker not in close_local.columns or close_local[ticker].dropna().empty
]

if missing_series:
    raise ValueError(
        f"Yahoo Finance returned no usable Close data for: {missing_series}"
    )

print("All required Yahoo Finance series contain data.")


## Currency normalization — essential for a European portfolio

The five defence stocks are not quoted in one currency:

- BAE Systems is quoted by Yahoo Finance in **GBp (pence sterling)**;
- Rheinmetall, Thales, and Leonardo are quoted in **EUR**;
- Saab is quoted in **SEK**.

Adding those raw prices together would be financially meaningless.

We therefore convert every security to **USD** using Yahoo Finance FX series before calculating portfolio wealth.

For a EUR security:

\[
Price^{USD}_t = Price^{EUR}_t \times EURUSD_t
\]

For BAE Systems, Yahoo reports pence, so we first divide by 100:

\[
Price^{USD}_t =
\frac{Price^{GBp}_t}{100}
\times GBPUSD_t
\]

This produces an **unhedged USD investor perspective**: both stock performance and currency movements affect returns.


In [ ]:
# Create one aligned daily FX panel from Yahoo Finance.
# Forward filling is appropriate for non-trading-day mismatches between
# stock exchanges and FX calendars; we cap no lookback here because the
# source data are daily and the analysis uses common market dates later.
fx = close_local[list(FX_TICKERS.values())].copy().ffill()

def convert_to_usd(price_df, securities):
    """
    Convert local-currency Yahoo prices to USD.

    Parameters
    ----------
    price_df : DataFrame
        Date x Ticker local-currency price matrix.
    securities : dict
        Security metadata containing Quote_Currency.

    Returns
    -------
    DataFrame
        Date x Ticker prices expressed in USD.
    """
    usd = pd.DataFrame(index=price_df.index)

    for ticker, meta in securities.items():
        currency = meta["Quote_Currency"]
        local_price = price_df[ticker].copy()

        if currency == "EUR":
            # EURUSD=X = USD received for one EUR.
            usd[ticker] = local_price * fx["EURUSD=X"]

        elif currency == "SEK":
            # SEKUSD=X = USD received for one SEK.
            usd[ticker] = local_price * fx["SEKUSD=X"]

        elif currency == "GBp":
            # Yahoo quotes London shares in pence.
            # Divide by 100 to convert pence -> pounds, then GBP -> USD.
            usd[ticker] = (local_price / 100.0) * fx["GBPUSD=X"]

        else:
            raise ValueError(
                f"Unsupported quote currency '{currency}' for {ticker}."
            )

    return usd


# Convert both Close and Adjusted Close because the portfolio reconstruction
# uses the initial Close and the dividend-adjusted growth path.
defence_close_usd = convert_to_usd(close_local, DEFENCE)
defence_adj_usd = convert_to_usd(adj_local, DEFENCE)

display(defence_close_usd.tail())


## Reconstruct Mary's portfolio

The case assumes Mary initially bought the **same number of shares** of each company.

That is **not equal weighting**.

If she buys one share of every company, initial dollar exposure depends on each company's USD-equivalent share price.

For each company:

\[
TR_{i,t}=
\frac{AdjPrice^{USD}_{i,t}}
{AdjPrice^{USD}_{i,0}}
\]

and the value of the initial one-share position evolves as:

\[
Wealth_{i,t}
=
ClosePrice^{USD}_{i,0}\times TR_{i,t}
\]

The total portfolio index is then normalized to 100.


In [ ]:
# Keep only dates on which all five defence securities have usable data.
# This prevents one missing series from silently changing the portfolio.
common_dates = defence_close_usd[DEFENCE_TICKERS].dropna().index
common_dates = common_dates.intersection(
    defence_adj_usd[DEFENCE_TICKERS].dropna().index
)

defence_close_usd = defence_close_usd.loc[common_dates, DEFENCE_TICKERS]
defence_adj_usd = defence_adj_usd.loc[common_dates, DEFENCE_TICKERS]

if defence_close_usd.empty:
    raise ValueError("No common trading history exists for all defence securities.")

# Use the first common observation as the portfolio inception date.
base_date = defence_close_usd.index.min()

initial_close_usd = defence_close_usd.loc[base_date]
initial_adj_usd = defence_adj_usd.loc[base_date]

# Adjusted-price growth represents the total-return path after
# incorporating dividends/corporate-action adjustments from Yahoo.
total_return_factor = defence_adj_usd.div(initial_adj_usd)

# One initial share of each company.
position_wealth_usd = total_return_factor.mul(initial_close_usd, axis=1)

# Sum the five USD position values to create portfolio wealth.
mary_wealth_usd = position_wealth_usd.sum(axis=1)

# Normalize portfolio wealth to an index starting at 100.
mary_index = (
    mary_wealth_usd / mary_wealth_usd.iloc[0] * 100
).rename("European Defence Portfolio")

# Show the economic capital weights implied by equal numbers of shares.
initial_weights = (
    initial_close_usd / initial_close_usd.sum()
).rename("Initial Capital Weight")

print("Portfolio base date:", base_date.date())
display(initial_weights.to_frame().style.format("{:.2%}"))
display(mary_index.head())


In [ ]:
# Plot the reconstructed defence portfolio as a normalized wealth index.
fig, ax = plt.subplots(figsize=(13, 7))

mary_index.plot(ax=ax, linewidth=2)

ax.set_title(
    "Mary's European Defence Portfolio — USD Total-Return Index (Base = 100)"
)
ax.set_xlabel("Date")
ax.set_ylabel("Portfolio Index")
ax.axhline(100, linestyle="--", linewidth=1)

plt.tight_layout()
plt.show()

print(
    f"Start index: {mary_index.iloc[0]:.2f}\n"
    f"Latest index: {mary_index.iloc[-1]:.2f}\n"
    f"Total wealth multiple: {mary_index.iloc[-1] / 100:.2f}x"
)


# Step 3 — Test the six-year capital-loss claim

Suppose Mary argues:

> **“If I hold my defence portfolio for at least six years, I cannot lose money.”**

Testing only calendar-year entry points would be weak.

Instead, we use month-end observations and test **every start/end combination with a holding period of at least six years**.

For each pair:

\[
HoldingPeriodReturn =
\frac{Index_{end}}{Index_{start}}-1
\]

If any observation is negative, the statement is historically false for this sample.

This remains a **historical test**, not a guarantee about future six-year periods.


In [ ]:
# Convert the daily portfolio index into month-end observations.
# Month-end sampling removes redundant daily entry dates while retaining
# a much stronger test than one observation per calendar year.
mary_monthly = mary_index.resample("ME").last().dropna()

records = []

# Compare every possible month-end start date with every end date
# at least 72 monthly observations later.
for start_pos in range(len(mary_monthly)):
    start_date = mary_monthly.index[start_pos]
    start_value = mary_monthly.iloc[start_pos]

    for end_pos in range(start_pos + 72, len(mary_monthly)):
        end_date = mary_monthly.index[end_pos]
        end_value = mary_monthly.iloc[end_pos]

        years = (end_date - start_date).days / 365.25

        if years >= 6:
            hpr = end_value / start_value - 1

            records.append({
                "Start": start_date,
                "End": end_date,
                "Years": years,
                "Holding_Period_Return": hpr,
            })

holding_periods = pd.DataFrame(records)

if holding_periods.empty:
    raise ValueError(
        "The downloaded history is too short to test holding periods >= 6 years."
    )

worst_period = holding_periods.loc[
    holding_periods["Holding_Period_Return"].idxmin()
]

negative_long_periods = holding_periods[
    holding_periods["Holding_Period_Return"] < 0
]

print(
    "Historical six-year claim:",
    "SUPPORTED in sample" if negative_long_periods.empty else "FALSE in sample"
)
print("\nWorst >=6-year period:")
display(
    worst_period.to_frame("Value").style.format({
        "Holding_Period_Return": "{:.2%}",
        "Years": "{:.2f}",
    })
)
print(
    "Negative >=6-year start/end combinations:",
    len(negative_long_periods)
)


In [ ]:
# Rolling six-year returns make the holding-period test visually intuitive.
rolling_6y = (
    mary_monthly / mary_monthly.shift(72) - 1
).dropna()

fig, ax = plt.subplots(figsize=(13, 6))
(rolling_6y * 100).plot(ax=ax)

ax.axhline(0, linestyle="--", linewidth=1)
ax.set_title("Rolling 6-Year Return — European Defence Portfolio")
ax.set_xlabel("End Date")
ax.set_ylabel("6-Year Return (%)")

plt.tight_layout()
plt.show()


# Step 4 — Build a Yahoo Finance diversification comparison

A concentrated defence portfolio should not be evaluated only against other defence companies.

We create a comparison universe of major European companies from different economic sectors:

- Technology — ASML
- Financials — Banco Santander
- Health Care — Sanofi
- Utilities — Iberdrola
- Consumer Discretionary — LVMH

All five are euro-quoted. Their Yahoo Finance **Adjusted Close** series are converted to USD using `EURUSD=X`.

We also use **EXSA.DE**, the iShares STOXX Europe 600 ETF, as the broad European market benchmark.


In [ ]:
# Convert any EUR-quoted Adjusted Close series to USD.
def eur_series_to_usd(adj_price_series):
    """Convert a EUR price series into unhedged USD using Yahoo EURUSD."""
    return adj_price_series * fx["EURUSD=X"]


# Build USD total-return price series for the five diversification companies.
diversifier_usd = pd.DataFrame(index=adj_local.index)

for ticker in DIVERSIFIER_TICKERS:
    diversifier_usd[ticker] = eur_series_to_usd(adj_local[ticker])

# Convert the European-market ETF benchmark from EUR to USD as well.
market_usd = eur_series_to_usd(adj_local[MARKET_TICKER]).rename("Market")

# Normalize every comparison series to 100 on the first common observation.
comparison_prices = pd.concat(
    [
        mary_index,
        diversifier_usd.rename(columns=DIVERSIFIERS),
        market_usd,
    ],
    axis=1,
).dropna()

comparison_norm = (
    comparison_prices / comparison_prices.iloc[0] * 100
)

display(comparison_norm.head())


In [ ]:
# Reshape the normalized panel into long format for Plotly.
plot_df = (
    comparison_norm
    .reset_index(names="Date")
    .melt(
        id_vars="Date",
        var_name="Asset",
        value_name="Index",
    )
)

# Interactive chart: hover across dates and compare normalized wealth paths.
fig = px.line(
    plot_df,
    x="Date",
    y="Index",
    color="Asset",
    title="European Defence vs Diversification Assets — USD Total Return, Base 100",
)

fig.update_layout(
    hovermode="x unified",
    yaxis_title="Normalized Index",
    xaxis_title="Date",
    legend_title="Asset",
)

fig.show()


# Step 5 — Risk, Return, Sharpe and Downside Risk

Daily simple return:

\[
R_t = \frac{P_t}{P_{t-1}}-1
\]

Annualized arithmetic return:

\[
\bar R_{annual}=252\bar R_{daily}
\]

Annualized volatility:

\[
\sigma_{annual}=\sigma_{daily}\sqrt{252}
\]

Sharpe Ratio:

\[
Sharpe=\frac{R-R_f}{\sigma}
\]

## Risk-free rate

Instead of hard-coding a rate, the notebook downloads Yahoo Finance ticker **`^IRX`**, the 13-week US Treasury Bill yield proxy.

Because the portfolio is evaluated in USD, this provides a consistent Yahoo-sourced cash-rate approximation. We use the average available yield over the common analysis period.

This is an analytical approximation, not an institutional curve construction.


In [ ]:
# Yahoo's ^IRX series is quoted as an annualized percentage yield.
# Convert it from percentage points (for example 4.50) to decimal (0.045).
rf_yield = close_local[RF_TICKER].dropna() / 100.0

# Restrict the risk-free series to the same broad historical window.
rf_yield = rf_yield.loc[
    comparison_prices.index.min():comparison_prices.index.max()
]

RF = float(rf_yield.mean())

print(f"Average Yahoo ^IRX risk-free proxy used: {RF:.2%}")

# Calculate daily simple returns from common USD total-return series.
returns = comparison_prices.pct_change().dropna()


def ann_risk_return(returns_df):
    """
    Calculate annualized arithmetic return and annualized volatility.
    """
    summary = returns_df.agg(["mean", "std"]).T
    summary.columns = ["Return", "Risk"]

    summary["Return"] = summary["Return"] * TRADING_DAYS
    summary["Risk"] = summary["Risk"] * np.sqrt(TRADING_DAYS)

    return summary


summary = ann_risk_return(returns)

# Sharpe Ratio compares annualized excess return with annualized volatility.
summary["Sharpe"] = (
    (summary["Return"] - RF)
    / summary["Risk"]
)

summary = summary.sort_values("Sharpe", ascending=False)

display(
    summary.style.format({
        "Return": "{:.2%}",
        "Risk": "{:.2%}",
        "Sharpe": "{:.3f}",
    })
)


In [ ]:
def max_drawdown_from_prices(price_series):
    """
    Calculate the worst peak-to-trough percentage loss.
    """
    wealth = price_series / price_series.iloc[0]
    running_peak = wealth.cummax()
    drawdown = wealth / running_peak - 1
    return drawdown.min()


def annualized_downside_deviation(return_series, threshold=0.0):
    """
    Annualize only returns below the chosen threshold.
    """
    downside = np.minimum(return_series - threshold, 0)
    return np.sqrt(np.mean(downside ** 2)) * np.sqrt(TRADING_DAYS)


# Add downside-focused metrics that are particularly relevant
# for a retired investor who cares about severe losses and recovery paths.
summary["Max_Drawdown"] = [
    max_drawdown_from_prices(comparison_prices[col])
    for col in summary.index
]

summary["Downside_Deviation"] = [
    annualized_downside_deviation(returns[col])
    for col in summary.index
]

summary["Sortino"] = (
    (summary["Return"] - RF)
    / summary["Downside_Deviation"]
)

display(
    summary[
        [
            "Return",
            "Risk",
            "Sharpe",
            "Max_Drawdown",
            "Downside_Deviation",
            "Sortino",
        ]
    ].style.format({
        "Return": "{:.2%}",
        "Risk": "{:.2%}",
        "Sharpe": "{:.3f}",
        "Max_Drawdown": "{:.2%}",
        "Downside_Deviation": "{:.2%}",
        "Sortino": "{:.3f}",
    })
)


In [ ]:
# Visualize the return/volatility trade-off.
fig, ax = plt.subplots(figsize=(11, 8))

ax.scatter(
    summary["Risk"],
    summary["Return"],
    s=90,
)

# Annotate every asset so the chart can be interpreted without a legend.
for name, row in summary.iterrows():
    ax.annotate(
        name,
        (row["Risk"], row["Return"]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8,
    )

ax.set_title("Risk–Return Map — Yahoo Finance, USD")
ax.set_xlabel("Annualized Volatility")
ax.set_ylabel("Annualized Arithmetic Return")

plt.tight_layout()
plt.show()


# Step 6 — Demonstrate the diversification effect

A common error is:

> **“If another asset has a lower standalone Sharpe Ratio than defence, adding it cannot improve the portfolio.”**

That is false because portfolio risk depends on **covariance**:

\[
\sigma_p^2=w^\top\Sigma w
\]

An asset with weaker standalone performance can still improve the overall portfolio if its return path is sufficiently different.

We simulate **50,000 long-only portfolios** across:

- Mary's European Defence Portfolio;
- ASML;
- Banco Santander;
- Sanofi;
- Iberdrola;
- LVMH.

We then search for:

1. the highest-Sharpe simulated portfolio;
2. the best portfolio with **no more volatility than the defence portfolio**.


In [ ]:
# Exclude the broad market benchmark from the optimization universe.
portfolio_assets = [
    "European Defence Portfolio",
    *list(DIVERSIFIERS.values()),
]

portfolio_returns = returns[portfolio_assets].dropna()

np.random.seed(SEED)

n_assets = len(portfolio_assets)

# Generate random positive weights and normalize every row to 100%.
weights = np.random.random(
    size=(N_PORTFOLIOS, n_assets)
)
weights = weights / weights.sum(axis=1, keepdims=True)

# Annualize the sample means and covariance matrix.
annual_mean = (
    portfolio_returns.mean().values
    * TRADING_DAYS
)

annual_cov = (
    portfolio_returns.cov().values
    * TRADING_DAYS
)

# Matrix multiplication gives expected portfolio return for all simulations.
portfolio_return = weights @ annual_mean

# Efficiently evaluate w'Σw for all simulated portfolios.
portfolio_var = np.einsum(
    "ij,jk,ik->i",
    weights,
    annual_cov,
    weights,
)

portfolio_risk = np.sqrt(portfolio_var)

portfolio_sharpe = (
    (portfolio_return - RF)
    / portfolio_risk
)

sim = pd.DataFrame({
    "Return": portfolio_return,
    "Risk": portfolio_risk,
    "Sharpe": portfolio_sharpe,
})

# Find the maximum-Sharpe portfolio in the simulation.
max_sharpe_idx = sim["Sharpe"].idxmax()

# Capture Mary's standalone defence-portfolio metrics.
def_return = summary.loc["European Defence Portfolio", "Return"]
def_risk = summary.loc["European Defence Portfolio", "Risk"]
def_sharpe = summary.loc["European Defence Portfolio", "Sharpe"]

# Restrict the search to portfolios no more volatile than Mary's defence sleeve.
eligible = sim[sim["Risk"] <= def_risk]

if eligible.empty:
    raise RuntimeError(
        "No simulated diversified portfolio met the defence-risk constraint."
    )

best_same_risk_idx = eligible["Sharpe"].idxmax()

# Convert winning weight arrays into labelled Series for interpretation.
max_sharpe_weights = pd.Series(
    weights[max_sharpe_idx],
    index=portfolio_assets,
    name="Max Sharpe Weight",
)

same_risk_weights = pd.Series(
    weights[best_same_risk_idx],
    index=portfolio_assets,
    name="Best <= Defence Risk Weight",
)

comparison = pd.DataFrame({
    "Defence Only": [
        def_return,
        def_risk,
        def_sharpe,
    ],
    "Max Sharpe (50k)": [
        sim.loc[max_sharpe_idx, "Return"],
        sim.loc[max_sharpe_idx, "Risk"],
        sim.loc[max_sharpe_idx, "Sharpe"],
    ],
    "Best Sharpe <= Defence Risk": [
        sim.loc[best_same_risk_idx, "Return"],
        sim.loc[best_same_risk_idx, "Risk"],
        sim.loc[best_same_risk_idx, "Sharpe"],
    ],
}, index=["Return", "Risk", "Sharpe"])

display(comparison.style.format("{:.2%}", subset=pd.IndexSlice[["Return", "Risk"], :]))
display(max_sharpe_weights.to_frame().style.format("{:.2%}"))
display(same_risk_weights.to_frame().style.format("{:.2%}"))


In [ ]:
# Plot the simulated portfolio opportunity set.
fig, ax = plt.subplots(figsize=(12, 8))

scatter = ax.scatter(
    sim["Risk"],
    sim["Return"],
    c=sim["Sharpe"],
    s=10,
    alpha=0.35,
)

# Mark the concentrated defence portfolio.
ax.scatter(
    def_risk,
    def_return,
    marker="X",
    s=180,
    label="European Defence Portfolio",
)

# Mark the best Sharpe portfolio subject to the same-risk constraint.
ax.scatter(
    sim.loc[best_same_risk_idx, "Risk"],
    sim.loc[best_same_risk_idx, "Return"],
    marker="*",
    s=220,
    label="Best Sharpe at <= Defence Risk",
)

# Mark the maximum-Sharpe portfolio from the entire simulation.
ax.scatter(
    sim.loc[max_sharpe_idx, "Risk"],
    sim.loc[max_sharpe_idx, "Return"],
    marker="D",
    s=120,
    label="Max Sharpe (50k)",
)

ax.set_title("50,000 Simulated Portfolios — Diversification Opportunity Set")
ax.set_xlabel("Annualized Risk")
ax.set_ylabel("Annualized Return")
ax.legend()

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("Sharpe Ratio")

plt.tight_layout()
plt.show()


## Correlation — the mechanism behind diversification


In [ ]:
# Correlation measures the degree to which daily returns move together.
# Lower or negative correlations create the mathematical opportunity
# for diversification to reduce total portfolio risk.
corr = portfolio_returns.corr()

fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(corr.values, aspect="auto")

ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticks(range(len(corr.index)))
ax.set_yticklabels(corr.index)

# Write the pairwise correlation inside every matrix cell.
for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        ax.text(
            j,
            i,
            f"{corr.iloc[i, j]:.2f}",
            ha="center",
            va="center",
            fontsize=7,
        )

ax.set_title("Return Correlation Matrix")
plt.colorbar(im, ax=ax, label="Correlation")
plt.tight_layout()
plt.show()


# Step 7 — CAPM: Alpha and Beta versus the European market

We use Yahoo Finance ticker **EXSA.DE** as the market proxy.

For asset \(i\):

\[
\beta_i=
\frac{Cov(R_i,R_M)}
{Var(R_M)}
\]

\[
CAPM_i=
R_f+\beta_i(R_M-R_f)
\]

\[
Alpha_i=
R_i-CAPM_i
\]

Interpretation:

- **Beta < 1** means the asset historically moved less than one-for-one with the market in systematic-risk terms.
- **Positive historical Alpha** means realized annualized return exceeded the CAPM-implied return over this sample.

Neither is a forecast.


In [ ]:
# Build one CAPM return panel using the same USD common-currency basis.
capm_prices = comparison_prices[
    ["European Defence Portfolio", *list(DIVERSIFIERS.values()), "Market"]
].dropna()

capm_returns = capm_prices.pct_change().dropna()

# Reuse the annualized return/risk helper.
capm_summary = ann_risk_return(capm_returns)

# Annualized covariance matrix is required for Beta and risk decomposition.
annual_cov_capm = (
    capm_returns.cov()
    * TRADING_DAYS
)

market_var = annual_cov_capm.loc["Market", "Market"]
market_return = capm_summary.loc["Market", "Return"]

# Standard Sharpe ratio.
capm_summary["Sharpe"] = (
    (capm_summary["Return"] - RF)
    / capm_summary["Risk"]
)

# Annualized variance of each asset.
capm_summary["Total_Risk_Var"] = (
    capm_returns.var()
    * TRADING_DAYS
)

# Beta = covariance with market / market variance.
capm_summary["Beta"] = (
    annual_cov_capm["Market"]
    / market_var
)

# CAPM risk decomposition.
capm_summary["Systematic_Risk"] = (
    capm_summary["Beta"] ** 2
    * market_var
)

capm_summary["Unsystematic_Risk"] = (
    capm_summary["Total_Risk_Var"]
    - capm_summary["Systematic_Risk"]
)

# Remove tiny floating-point artifacts around zero.
capm_summary.loc[
    capm_summary["Unsystematic_Risk"].abs() < 1e-12,
    "Unsystematic_Risk"
] = 0.0

# CAPM expected return under the historical sample inputs.
capm_summary["CAPM_Return"] = (
    RF
    + capm_summary["Beta"] * (market_return - RF)
)

# Historical Alpha = realized return minus CAPM-implied return.
capm_summary["Alpha"] = (
    capm_summary["Return"]
    - capm_summary["CAPM_Return"]
)

# Additional diagnostics.
capm_summary["Market_Correlation"] = (
    capm_returns.corr()["Market"]
)

capm_summary["R_squared"] = (
    capm_summary["Systematic_Risk"]
    / capm_summary["Total_Risk_Var"]
)

display(
    capm_summary.style.format({
        "Return": "{:.2%}",
        "Risk": "{:.2%}",
        "Sharpe": "{:.3f}",
        "Total_Risk_Var": "{:.5f}",
        "Systematic_Risk": "{:.5f}",
        "Unsystematic_Risk": "{:.5f}",
        "Beta": "{:.3f}",
        "CAPM_Return": "{:.2%}",
        "Alpha": "{:+.2%}",
        "Market_Correlation": "{:.3f}",
        "R_squared": "{:.2%}",
    })
)


## Identify diversification candidates satisfying a simple Alpha/Beta screen


In [ ]:
# Remove the benchmark itself and apply a transparent historical screen.
candidate_assets = (
    capm_summary
    .drop(index="Market")
    .query("Alpha > 0 and Beta < 1")
    .sort_values("Alpha", ascending=False)
)

print("Assets satisfying historical Alpha > 0 and Beta < 1:")
display(
    candidate_assets[
        [
            "Return",
            "Risk",
            "Sharpe",
            "Beta",
            "CAPM_Return",
            "Alpha",
            "Market_Correlation",
        ]
    ].style.format({
        "Return": "{:.2%}",
        "Risk": "{:.2%}",
        "Sharpe": "{:.3f}",
        "Beta": "{:.3f}",
        "CAPM_Return": "{:.2%}",
        "Alpha": "{:+.2%}",
        "Market_Correlation": "{:.3f}",
    })
)


## Security Market Line


In [ ]:
# Create a range of Beta values wide enough to contain the observed assets.
beta_grid = np.linspace(
    0,
    max(1.8, capm_summary["Beta"].max() + 0.20),
    200,
)

# Security Market Line implied by the sample market return and RF proxy.
sml = (
    RF
    + beta_grid * (market_return - RF)
)

fig, ax = plt.subplots(figsize=(12, 8))

ax.plot(
    beta_grid,
    sml,
    label="Security Market Line",
)

ax.scatter(
    capm_summary["Beta"],
    capm_summary["Return"],
    s=80,
)

for name, row in capm_summary.iterrows():
    ax.annotate(
        name,
        (row["Beta"], row["Return"]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8,
    )

ax.set_title("Security Market Line — European Portfolio Analysis")
ax.set_xlabel("Beta")
ax.set_ylabel("Annualized Return")
ax.legend()

plt.tight_layout()
plt.show()


# Step 8 — Robustness: rolling Beta

A single full-period Beta estimate can hide major regime changes.

Defence companies can behave differently during:

- low defence-spending regimes;
- geopolitical shocks;
- fiscal expansion;
- recession;
- broad equity sell-offs;
- major procurement cycles.

We therefore calculate **rolling 252-trading-day Beta**:

\[
\beta_{i,t}^{rolling} =
\frac{Cov_{252}(R_i,R_M)}
{Var_{252}(R_M)}
\]

If Beta changes materially through time, a single historical Beta should not be treated as a stable characteristic.


In [ ]:
ROLLING_WINDOW = 252

# Rolling market variance is the denominator for every asset Beta.
rolling_market_var = (
    capm_returns["Market"]
    .rolling(ROLLING_WINDOW)
    .var()
)

rolling_beta = pd.DataFrame(index=capm_returns.index)

# Calculate rolling covariance with the market and divide by
# the market's rolling variance.
for col in capm_returns.columns:
    rolling_cov = (
        capm_returns[col]
        .rolling(ROLLING_WINDOW)
        .cov(capm_returns["Market"])
    )

    rolling_beta[col] = (
        rolling_cov / rolling_market_var
    )

fig, ax = plt.subplots(figsize=(13, 7))

# Plot the concentrated defence portfolio and any assets that pass
# the historical Alpha/Beta screen.
plot_cols = ["European Defence Portfolio"]

for col in candidate_assets.index:
    if col not in plot_cols and col != "Market":
        plot_cols.append(col)

for col in plot_cols:
    if col in rolling_beta.columns:
        ax.plot(
            rolling_beta.index,
            rolling_beta[col],
            label=col,
        )

ax.axhline(1, linestyle="--", linewidth=1)
ax.set_title("Rolling 252-Day Beta — Stability Check")
ax.set_xlabel("Date")
ax.set_ylabel("Beta")
ax.legend()

plt.tight_layout()
plt.show()


# Step 9 — Production hand-off view

This notebook is a **decision-grade research prototype**, not a production trading system.

A production implementation should separate the current notebook into explicit components:

### 1. Market-data ingestion
**Input:** Yahoo/vendor symbols, date range, fields  
**Production target:** scheduled ingestion job → raw/bronze storage

### 2. Reference-data layer
**Input:** ticker, company, exchange, quote currency, sector, country  
**Production target:** governed security master

### 3. FX normalization
**Input:** local prices + FX rates  
**Output:** common-currency prices with conversion timestamp and lineage

### 4. Total-return transformation
**Input:** adjusted prices / corporate actions  
**Output:** validated total-return series

### 5. Portfolio engine
**Input:** holdings/weights + price series  
**Output:** NAV, P&L, returns, exposures

### 6. Risk analytics
**Output:** volatility, drawdown, Sharpe, Sortino, covariance, Beta, Alpha

### 7. Optimization layer
**Output:** candidate portfolios subject to explicit investment constraints

### 8. Monitoring
Production controls should include:

- stale-price checks;
- missing-ticker alerts;
- abnormal return/outlier checks;
- FX freshness;
- corporate-action reconciliation;
- benchmark availability;
- schema checks;
- reproducibility/versioning;
- model-input and output logs.

The notebook therefore gives engineering, product, risk, and investment teams a **clear preview of the production analytical contract**.


# Investment Committee Summary — How to Think Like a Strategist

### 1. Diagnosis
Mary has exposure to five high-profile European defence businesses, but country diversification inside one industry does **not eliminate thematic concentration**.

### 2. Data discipline
All market inputs in this notebook are requested from **Yahoo Finance via `yfinance`**:

- security prices;
- adjusted prices;
- FX;
- broad European benchmark;
- risk-free proxy.

No static market-data CSV/XLSX file is required.

### 3. Economic correctness
Multi-currency securities are converted to a common USD base **before portfolio aggregation**.

This prevents the common analytical error of adding EUR, GBp, and SEK prices together.

### 4. Evidence
Evaluate:

- total-return path;
- six-year holding-period outcomes;
- volatility and Sharpe;
- maximum drawdown;
- downside deviation and Sortino;
- correlations;
- simulated portfolio opportunity set;
- CAPM Beta and Alpha;
- rolling-Beta stability.

### 5. Diversification logic
A diversification asset does not need to outperform defence on its own to improve the overall portfolio.

\[
Portfolio\ Risk \neq
Weighted\ Average\ of\ Standalone\ Risks
\]

**Covariance is the mechanism.**

### 6. Explicit uncertainty
Do not claim that:

- historical defence returns will repeat;
- a six-year historical pattern guarantees future capital protection;
- historical Alpha will persist;
- Beta is stable.

The professional conclusion is conditional:

> **“The analysis identifies how the concentrated European defence sleeve behaved historically and which diversified combinations improved the observed risk/return trade-off. These are decision inputs, not guaranteed forecasts.”**

---

# Final analyst checklist

- [ ] Five European defence leaders explicitly defined
- [ ] All external market data requested from Yahoo Finance
- [ ] No dependency on listings/sector/S&P CSV files
- [ ] Local currencies converted to USD before aggregation
- [ ] GBp correctly converted to GBP for BAE Systems
- [ ] Adjusted Close used for total-return growth
- [ ] Equal-number-of-shares assumption distinguished from equal-weighting
- [ ] Six-year claim tested over many entry dates
- [ ] Diversification universe built from other European sectors
- [ ] Risk/return/Sharpe calculated
- [ ] Maximum drawdown, downside deviation, and Sortino calculated
- [ ] 50,000 portfolios simulated with seed 111
- [ ] Correlation mechanism visualized
- [ ] CAPM Beta and Alpha calculated against EXSA.DE
- [ ] Rolling Beta used as a robustness check
- [ ] Production hand-off architecture documented
